[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Toan-ten/Arxiv-Project-Group10/blob/feat%2Fm2-colab-benchmark/notebooks/m2/m2_colab_benchmark.ipynb)

# Benchmark M2 — Group 10 (CPU-only)

Notebook độc lập với export/recovery campaign. Parser đã sửa `nocite`, không dùng bản lỗi `072ee384`.
Chạy Configuration → Setup → Plan → Smoke một cohort → đọc audit → Run range. Không tự chạy toàn corpus.
Input chỉ đọc; output là campaign benchmark riêng. Không có API keys, crawler, Google Drive download hay Hugging Face upload.
Kết quả là quan sát thực tế, không bảo đảm mọi LaTeX parse được. Partial run không đại diện 200 cohort.

In [ ]:
# Một nơi cấu hình. STUDENT_ID chỉ đặt output của người vận hành.
STUDENT_ID = "24120232"
START_COHORT = 101
END_COHORT = 150
INPUT_DRIVE_DIR = f"/content/drive/MyDrive/M1-pipeline-optimization-full/{STUDENT_ID}/archives"
OUTPUT_DRIVE_DIR = f"/content/drive/MyDrive/ArxivGroup10/{STUDENT_ID}/m2-benchmark"
# 1 cohort: start=end; 50 cohort: 101–150; 200: 1–200 + input parent của cả nhóm.
# Mỗi cohort: .tar.gz + .manifest.json + .tar.gz.sha256, cùng thư mục.


## Setup — mount Drive và tải đúng release

Chọn CPU trong Runtime settings. Ô này chỉ mount Drive, tải source code/dependencies vào `/content` và kiểm tra runtime.
Không chạy parser, migration hay ghi campaign. Folder chia sẻ: thêm shortcut vào MyDrive hoặc dùng đường dẫn `/content/drive/Shareddrives/...` nhìn thấy được sau mount.
Không cần mở hoặc đọc `.env`.

In [ ]:
from google.colab import drive
from pathlib import Path
import json
import os
import subprocess
import sys
import uuid

drive.mount("/content/drive")
RELEASE_COMMIT = "f5562be0477ed844121bdd429b3c974dfd9cd41c"
PARSER_FINGERPRINT = "4e88c7fb0d3463c4c6815b5afff3f64066815068dd4c34db1e92528966d60b70"
if os.environ.get("COLAB_GPU") == "1" or os.environ.get("COLAB_TPU_ADDR"):
    raise RuntimeError("Chọn runtime CPU-only rồi chạy lại Setup")
CODE_DIR = Path("/content/group10-m2-benchmark-" + RELEASE_COMMIT[:12])
if not CODE_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", "https://github.com/Toan-ten/Arxiv-Project-Group10.git", str(CODE_DIR)], check=True)
    subprocess.run(["git", "-C", str(CODE_DIR), "checkout", "--detach", RELEASE_COMMIT], check=True)
head = subprocess.check_output(["git", "-C", str(CODE_DIR), "rev-parse", "HEAD"], text=True).strip()
if head != RELEASE_COMMIT or subprocess.check_output(["git", "-C", str(CODE_DIR), "status", "--porcelain"], text=True).strip():
    raise RuntimeError("Checkout không đúng release hoặc có local edits; không tự reset")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(CODE_DIR / "requirements-m2-benchmark.txt")], check=True)
SCRATCH = "/content/group10-m2-benchmark-scratch-" + RELEASE_COMMIT[:12]
LOCAL_CONFIG = Path("/content/group10-m2-benchmark-config.json")
print("Release:", RELEASE_COMMIT, "Python:", sys.version.split()[0])


In [ ]:
def current_config():
    # Đọc giá trị hiện tại: sửa Configuration xong không bị capture giá trị cũ.
    return dict(repo=str(CODE_DIR), release_commit=RELEASE_COMMIT,
                start_cohort=START_COHORT, end_cohort=END_COHORT,
                input_drive_dir=INPUT_DRIVE_DIR, output_drive_dir=OUTPUT_DRIVE_DIR,
                scratch=SCRATCH)

def benchmark_command(action):
    LOCAL_CONFIG.write_text(json.dumps(current_config(), ensure_ascii=False), encoding="utf8")
    return [sys.executable, "-X", "utf8", "-B", "-m", "scripts.m2_colab_benchmark",
            "--config", str(LOCAL_CONFIG), "--action", action]

def plan():
    return json.loads(subprocess.check_output(benchmark_command("plan"), cwd=CODE_DIR, text=True))

def run_benchmark(action):
    info = plan()
    print("Operator:", STUDENT_ID, "Scope:", START_COHORT, END_COHORT)
    print("Campaign:", info["campaign"])
    print("Existing markers (chưa re-audit):", info["existing_completions"])
    if action == "run":
        first = Path(info["campaign"]) / "completed" / (info["cohorts"][0] + ".json")
        if not first.is_file():
            raise RuntimeError("Chạy Smoke một cohort và kiểm tra audit trước Run range")
    result = subprocess.run(benchmark_command(action), cwd=CODE_DIR)
    if result.returncode:
        raise RuntimeError("Benchmark dừng an toàn; xem diagnostic/LATEST.json, không retry vô hạn")


## Plan — chỉ đọc

Xác nhận input/output và phạm vi. Plan chưa chứng nhận các completion markers.
Thay range hoặc release tạo campaign khác; giữ nguyên cấu hình để Resume. Input path phải tồn tại, không tự tạo dữ liệu.

In [ ]:
print(json.dumps(plan(), ensure_ascii=False, indent=2))

## Smoke — ghi campaign benchmark mới, chỉ một cohort

Ô này đọc/kiểm checksum archive, extract vào scratch local, parse mọi version cùng publication, ghi Parquet/checkpoints/reports mới trên Drive.
Không chạm campaign export cũ. Lỗi parse cô lập được giữ trong CSV; audit failure dừng cohort và giữ diagnostic.
Đọc `benchmark_report.md`, `paper_results.csv`, `m2_errors.csv` và `audit_outcome` trước khi chạy tiếp.

In [ ]:
run_benchmark("smoke")

## Run range / Resume — có ghi output benchmark

Chỉ chạy khi đã xem Smoke. Chạy lại ô này để Resume cùng campaign: checkpoint verified được skip sau re-audit, cohort chưa hoàn tất dùng cache native đúng fingerprint.
Sau runtime mới: Configuration → Setup → controller → Plan → ô này; không cần chạy lại parser cho cohort đã verified.
Khi input/audit lỗi: dừng, xem lỗi; không sửa manifest/checkpoint để vượt guard.

In [ ]:
run_benchmark("run")

## Xuất lại báo cáo — không gọi parser, có ghi snapshot báo cáo mới

Dùng khi cần xem partial run. Mỗi snapshot có checksums và bốn PNG; `LATEST.json` trỏ snapshot hoàn chỉnh.
Không cộng version-associated counts thành union table totals. Runtime theo version chỉ đo prepare/render worker; runtime tổng có input/read/write/audit.
Đọc mẫu số trong JSON/Markdown. Nếu runtime bị kill, thời gian chưa ghi không được bịa; throughput có thể null.

In [ ]:
run_benchmark("report")
info = plan()
latest = Path(info["campaign"]) / "LATEST.json"
if latest.is_file():
    print(latest.read_text(encoding="utf8"))

## Chỉ khi runtime cũ bị kill: kiểm tra locks (chỉ đọc)

Không chạy đồng thời hai runtime cùng campaign. Nếu còn lock, phải xác nhận process cũ đã dừng trước khi archive lock.
Hàm archive dưới đây chỉ chạy khi bạn gọi thủ công và nhập đúng run ID; không tự động gọi khi Run All.
Không đụng input, export/recovery hoặc checkpoint payload.

In [ ]:
def inspect_locks():
    root = Path(plan()["campaign"])
    if not root.exists():
        return []
    locks = sorted(p for p in root.rglob("*") if p.name in {".writer.lock", ".export-writer.lock"})
    for p in locks:
        if p.is_symlink() or not p.resolve().is_relative_to(root.resolve()):
            raise RuntimeError("Lock path không an toàn; dừng review")
        print(p, p.read_text(encoding="utf8", errors="replace"))
    return locks

def archive_stale_locks():
    root = Path(plan()["campaign"])
    locks = inspect_locks()
    if not locks:
        print("Không có lock")
        return
    answer = input("CHỈ khi runtime/process cũ đã dừng: nhập run ID để archive locks: ")
    if answer != root.name:
        raise RuntimeError("Không xác nhận; giữ nguyên locks")
    for p in locks:
        p.rename(p.with_name(p.name + ".reviewed-" + uuid.uuid4().hex))
        print("Đã archive lock evidence:", p)

inspect_locks()
# Nếu và chỉ nếu runtime cũ đã dừng: tự gọi archive_stale_locks() trong ô riêng.


## Bàn giao Poster

Outputs: `benchmark_summary.json/.csv`, `paper_results.csv`, `m2_errors.csv`, `error_statistics.csv`, `benchmark_report.md`, bốn PNG và `report_checksums.json` trong reports snapshot.
Không đưa fixture offline vào số liệu Poster. Chỉ kết luận trên phạm vi quan sát, nêu SHA/môi trường/audit và partial coverage.
Chưa có đo RAM/disk mới trong notebook này; không thay số liệu đó bằng lifetime RSS process cha. Không tự công bố final performance benchmark.
Hướng dẫn chi tiết: `docs/m2_colab_benchmark.md`.